# Cross-Polarisation Explorer

Analyse the polarisation response of a simulated structure by comparing
**cross**, **parallel**, and **total** reflectance spectra from the same run.

**Workflow**
1. Run `main.py` with *polarisation analysis* enabled (options 1 → 1 cross,
   1 → 2 parallel, 1 → 3 total) for the same run name. This produces
   `{run}_cross.csv`, `{run}_parallel.csv`, and `{run}_total.csv` in the run folder.
2. Fill in `POL_SPECIES` and `POL_RUN` below and run all cells.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
# ── Global configuration ───────────────────────────────────────────────────
RESULTS_DIR  = Path("../results")
WL_MIN       = 380    # nm
WL_MAX       = 780    # nm
SAVE_FIGURES = False  # True → save figures as PNG in the run folder

## 1 · Discover runs with polarisation data

Run the cell below to see which runs already have `cross`, `parallel`, or `total` CSV files.

In [ ]:
def scan_polarization_runs(results_dir: Path) -> pd.DataFrame:
    """
    Find all runs that have at least one polarisation-tagged CSV
    (cross, parallel, or total).  Returns a summary DataFrame with boolean
    columns has_cross, has_parallel, has_total.
    """
    registry: dict = {}
    for csv_file in sorted(results_dir.rglob("*.csv")):
        parts = csv_file.relative_to(results_dir).parts
        if len(parts) < 3:
            continue
        species, run = parts[0], parts[1]
        stem = csv_file.stem
        tag  = stem[len(run) + 1:] if stem.startswith(run + "_") else stem
        if tag not in ("cross", "parallel", "total"):
            continue
        key = (species, run)
        if key not in registry:
            registry[key] = {"species": species, "run": run,
                             "has_cross": False, "has_parallel": False, "has_total": False}
        registry[key][f"has_{tag}"] = True

    if not registry:
        return pd.DataFrame(columns=["species", "run", "has_cross", "has_parallel", "has_total"])
    return pd.DataFrame(registry.values()).reset_index(drop=True)


pol_catalog = scan_polarization_runs(RESULTS_DIR)

if pol_catalog.empty:
    print("[INFO] No runs with polarisation data found yet.")
    print("       Run main.py and choose the cross / parallel / total analysis options.")
else:
    print("Runs with polarisation data:\n")
    print(pol_catalog.to_string(index=False))

## 2 · Cross vs Parallel vs Total

Set `POL_SPECIES` and `POL_RUN`, then run the plot cell.

In [ ]:
# ── Selection ─────────────────────────────────────────────────────────────
POL_SPECIES = "menelaus"   # species folder name
POL_RUN     = "run1"       # experiment name used in main.py

In [ ]:
# Styling for each polarisation tag
_POL_STYLE = {
    "total":    {"color": "#444444", "linestyle": "-",  "label": "Total"},
    "parallel": {"color": "#1f77b4", "linestyle": "-",  "label": "Parallel"},
    "cross":    {"color": "#d62728", "linestyle": "--", "label": "Cross"},
}

def plot_cross_parallel(results_dir, species, run,
                        wl_min=WL_MIN, wl_max=WL_MAX, save=SAVE_FIGURES):
    """Overlay cross, parallel, and total spectra for one run."""
    run_dir = results_dir / species / run
    curves  = {}
    for tag, style in _POL_STYLE.items():
        path = run_dir / f"{run}_{tag}.csv"
        if path.exists():
            curves[tag] = (pd.read_csv(path).sort_values("wavelength_nm"), style)

    if not curves:
        print(f"[ERROR] No polarisation CSV files found in {run_dir}")
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    r_max = max(df["reflectance"].max() for df, _ in curves.values())

    for tag, (df, style) in curves.items():
        win = (df["wavelength_nm"] >= wl_min) & (df["wavelength_nm"] <= wl_max)
        if win.any():
            idx_peak = df[win]["reflectance"].idxmax()
            peak_wl  = df.loc[idx_peak, "wavelength_nm"]
            ax.axvline(peak_wl, color=style["color"],
                       linestyle=":", linewidth=1.2, alpha=0.5)
            ax.text(peak_wl + 4, r_max * 0.92,
                    f"{peak_wl:.0f} nm", color=style["color"],
                    fontsize=11, va="top", rotation=90)

        ax.plot(df["wavelength_nm"], df["reflectance"],
                color=style["color"], linestyle=style["linestyle"],
                linewidth=2.2, label=style["label"])

    ax.set_xlabel("Wavelength (nm)", fontsize=14)
    ax.set_ylabel("Reflectance",     fontsize=14)
    ax.set_title(fr"$\it{{{species}}}$  ·  {run} — Polarisation Analysis", fontsize=14)
    ax.legend(fontsize=12)
    ax.set_xlim(wl_min, wl_max)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout()

    if save:
        out = results_dir / species / run / f"{run}_polarisation.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"[OK] Saved: {out}")

    plt.show()


plot_cross_parallel(RESULTS_DIR, POL_SPECIES, POL_RUN)

## 3 · Cross / Total polarisation ratio

The ratio $R_{cross}(\lambda) / R_{total}(\lambda)$ quantifies how much of the reflected
light survives crossed polarisers at each wavelength.  A high ratio at the reflectance
peak indicates strong structural iridescence.

Requires both `{run}_cross.csv` and `{run}_total.csv` to be present.

In [ ]:
def plot_polarization_ratio(results_dir, species, run,
                             wl_min=WL_MIN, wl_max=WL_MAX, save=SAVE_FIGURES):
    """Plot R_cross / R_total as a function of wavelength."""
    run_dir     = results_dir / species / run
    path_cross  = run_dir / f"{run}_cross.csv"
    path_total  = run_dir / f"{run}_total.csv"

    if not path_cross.exists() or not path_total.exists():
        missing = [p.name for p in [path_cross, path_total] if not p.exists()]
        print(f"[ERROR] Missing file(s): {missing}")
        return

    df_cross = pd.read_csv(path_cross).sort_values("wavelength_nm")
    df_total = pd.read_csv(path_total).sort_values("wavelength_nm")

    # Interpolate cross onto the total wavelength grid
    wl      = df_total["wavelength_nm"].values
    r_cross = np.interp(wl, df_cross["wavelength_nm"].values, df_cross["reflectance"].values)
    r_total = df_total["reflectance"].values

    ratio = np.divide(r_cross, r_total,
                      out=np.zeros_like(r_cross),
                      where=r_total > 1e-6)

    mask = (wl >= wl_min) & (wl <= wl_max)

    # Peak of the ratio
    idx_peak = np.argmax(ratio[mask])
    peak_wl  = wl[mask][idx_peak]
    peak_val = ratio[mask][idx_peak]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(wl[mask], ratio[mask], linewidth=2, color="#9467bd")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="50 % threshold")
    ax.scatter(peak_wl, peak_val, color="#9467bd", zorder=6, s=60,
               label=f"Peak  {peak_wl:.0f} nm  ({peak_val:.3f})")

    ax.set_xlabel("Wavelength (nm)",    fontsize=14)
    ax.set_ylabel("Cross / Total",      fontsize=14)
    ax.set_title(fr"$\it{{{species}}}$  ·  {run} — Cross-polarisation ratio", fontsize=14)
    ax.legend(fontsize=11)
    ax.set_xlim(wl_min, wl_max)
    ax.set_ylim(0, 1.05)
    ax.tick_params(labelsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout()

    if save:
        out = results_dir / species / run / f"{run}_polarisation_ratio.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"[OK] Saved: {out}")

    plt.show()


plot_polarization_ratio(RESULTS_DIR, POL_SPECIES, POL_RUN)

## 4 · Multi-run polarisation comparison

Compare the **cross** spectrum across several runs (e.g. a parameter sweep) on the same axes.

In [ ]:
# ── Selection ─────────────────────────────────────────────────────────────
# One entry per run: (species, run_name, tag, legend_label)
COMPARE_RUNS = [
    ("menelaus", "run1", "cross",    "run1 – cross"),
    ("menelaus", "run1", "parallel", "run1 – parallel"),
    # ("menelaus", "run2", "cross",  "run2 – cross"),
]

In [ ]:
import matplotlib.cm as cm

def plot_polarization_comparison(results_dir, compare_runs,
                                  wl_min=WL_MIN, wl_max=WL_MAX, save_path=None):
    """Overlay any mix of cross/parallel/total spectra from multiple runs."""
    palette = list(cm.tab10.colors)
    fig, ax = plt.subplots(figsize=(10, 5))
    any_loaded = False

    for i, (species, run, tag, label) in enumerate(compare_runs):
        path = results_dir / species / run / f"{run}_{tag}.csv"
        if not path.exists():
            print(f"[WARN] Not found: {path.name} — skipping.")
            continue

        df    = pd.read_csv(path).sort_values("wavelength_nm")
        color = palette[i % len(palette)]
        ls    = "--" if tag == "cross" else "-"
        any_loaded = True

        ax.plot(df["wavelength_nm"], df["reflectance"],
                linewidth=2, color=color, linestyle=ls, label=label)

        win = (df["wavelength_nm"] >= wl_min) & (df["wavelength_nm"] <= wl_max)
        if win.any():
            idx_peak = df[win]["reflectance"].idxmax()
            ax.axvline(df.loc[idx_peak, "wavelength_nm"],
                       color=color, linestyle=":", linewidth=1.2, alpha=0.5)

    if not any_loaded:
        print("[INFO] No data loaded — check species / run / tag values.")
        plt.close(fig)
        return

    ax.set_xlabel("Wavelength (nm)", fontsize=14)
    ax.set_ylabel("Reflectance",     fontsize=14)
    ax.set_title("Polarisation Comparison", fontsize=14)
    ax.legend(fontsize=11)
    ax.set_xlim(wl_min, wl_max)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[OK] Saved: {save_path}")

    plt.show()


plot_polarization_comparison(RESULTS_DIR, COMPARE_RUNS)